# Layout & Buttons: feature tests

Manual tests for the `feat/layout_btns` features:

1. [Collapsible sections](#1.-Collapsible-sections): toggles, titles, starting state
2. [`hide_info_card`](#2.-hide_info_card)
3. [Frequency range](#3.-Frequency-range): `freq_min` / `freq_max`, kept across clips
4. [Buttons](#4.-Buttons): built-in icons, colours, sizes, `initial_*`, toggles, submit, images, dynamic forms
5. [Validation](#5.-Validation)
6. [Output](#6.-Output)

Everything shared lives in the base config ([`config/layout-and-buttons.yaml`](annotator_config/config/layout-and-buttons.yaml))
and form ([`forms/layout-and-buttons.yaml`](annotator_config/forms/layout-and-buttons.yaml)).
Each test passes **only the settings it is testing** as a `test` dict, so what is being tested is
right there in the cell. Each section lists what to check.

In [1]:
from jupyter_bioacoustic import BioacousticAnnotator

CONFIG = 'annotator_config/config/layout-and-buttons.yaml'


def jba(**test):
    """Open the annotator on the base config plus ``test``; print any validation problems."""
    ba = BioacousticAnnotator(config=CONFIG, **test)
    result = ba.validate()
    for msg in result['errors']:
        print('❌', msg)
    for msg in result['warnings']:
        print('⚠', msg)
    ba.open()
    return ba

---
## 1. Collapsible sections

### 1.a Defaults

`test = {}`: the base config only. The base form is a quick review: ✓ / ✗ / ? radio buttons,
a 3-state call-type toggle and a one-click ⚑ flag + submit button.

Check:
- [ ] the **info card is at the top of the playlist** (above the filter bar) and starts with **▾** (no text)
- [ ] the **play row is at the top of the spectrogram** (above Type / Res) and starts with **▾** (no text)
- [ ] the form's **title row** (QUICK REVIEW … 0 / N) starts with **▾**
- [ ] info card ▾ → hides the filter bar + clip table; the info card stays (Prev / Next still work)
- [ ] play row ▾ → hides the controls, view bar and canvas; the play row stays (▶ still plays)
- [ ] form ▾ → hides everything but that title row (progress still updates)
- [ ] forms with no title / progress row (the section 4 tests) put ▾ in the Skip / Submit row instead, and keep that row
- [ ] each ▸ expands again; the spectrogram redraws at the right size after expanding
- [ ] Tab to a ▾ and press Enter: it toggles (same as a click) and does **not** submit

In [2]:
test = {}
ba_defaults = jba(**test)

<IPython.core.display.Javascript object>

### 1.b Toggle titles

`playlist_title` / `spectrogram_title` default to no text. `True` gives the standard name; a string sets it.

Check:
- [ ] info card starts with **▾ Playlist**
- [ ] play row starts with **▾ Spectrogram (kHz)**

In [20]:
test = {
    'playlist_title': False,
    # 'spectrogram_title': 'Spectrogram (kHz)',
}
jba(**test);

<IPython.core.display.Javascript object>

### 1.c Starting collapsed

Check:
- [ ] playlist, spectrogram and form all start collapsed (▸)
- [ ] expanding the spectrogram draws it at full size (it loaded while hidden)
- [ ] expanding the form shows all of it

In [15]:
test = {
    'playlist_open': False,
    'spectrogram_open': False,
    'form_open': False,
}
jba(**test);

<IPython.core.display.Javascript object>

### 1.d Collapsed form in the reviewed view

With `duplicate_entries: false`, a submitted clip shows its saved answers (the "reviewed view")
instead of the form. Uses its own output file.

Check:
- [ ] submit a clip, then **◀ Prev** back to it: the reviewed view shows, with ▾ at the start of its REVIEWED title
- [ ] collapse the form: only the REVIEWED row stays
- [ ] **Next** to an unreviewed clip: the form is rebuilt, still collapsed, ▸ in the QUICK REVIEW row
- [ ] expand: the full form is back

In [ ]:
test = {
    'duplicate_entries': False,
    'output_path': 'outputs/layout-and-buttons.reviewed.csv',
}
jba(**test);

---
## 2. `hide_info_card`

With the info card hidden, the playlist toggle gets a slim row of its own in the same place.

Check:
- [ ] no info card (no time range, no Prev / Next in that row)
- [ ] a slim row with **▾ Playlist** above the filter bar; it collapses the playlist
- [ ] second cell (no title): the slim row has just ▾
- [ ] the inline widget isn't taller than it needs to be

In [ ]:
test = {
    'hide_info_card': True,
    'playlist_title': True,
}
jba(**test);

In [ ]:
test = {'hide_info_card': True}
jba(**test);

---
## 3. Frequency range

`freq_min` / `freq_max` (kHz) set the frequency window the spectrogram opens with, and
the one **Reset** (or the `0` key) goes back to. Whatever frequency zoom you make is kept
when you move to another clip or change Type / Res; the time zoom resets per clip.

### 3.a `freq_min: 1`, `freq_max: 8`

Check:
- [ ] the first clip opens showing 1–8 kHz (Freq min / max boxes read 1.0 / 8.0)
- [ ] **Next ▶**: still 1–8 kHz
- [ ] type 2 / 5 into the Freq boxes (Enter) → **Next ▶** → still 2–5 kHz
- [ ] zoom box (⬚) on a band → **Next ▶** → same band; the time axis is back to the whole clip
- [ ] switch Type Linear → Mel and **Update**: the same Hz band is shown on the mel scale
- [ ] **Reset**: back to 1–8 kHz (not the full range)

In [20]:
test = {
    'freq_min': 1,
    'freq_max': 8,
}
jba(**test);

<IPython.core.display.Javascript object>

### 3.b Only `freq_max`; and no setting at all

Check:
- [ ] first cell opens at 0–6 kHz
- [ ] second cell (no setting) opens at the full range; zoom to a band, **Next ▶**: the band is kept; **Reset**: full range

In [25]:
test = {'freq_max': 6}
jba(**test);

<IPython.core.display.Javascript object>

In [ ]:
test = {}
jba(**test);

### 3.c Bad values

Each of these should raise a `ValueError` (printed below, no widget).

In [ ]:
for bad in ({'freq_min': 'low'}, {'freq_max': -1}, {'freq_min': 9, 'freq_max': 2}):
    try:
        BioacousticAnnotator(config=CONFIG, **bad)
        print('✗ no error for', bad)
    except ValueError as e:
        print('✓', bad, '->', e)

---
## 4. Buttons

Each test replaces the base form with its own `form_config`, so the cell shows the whole form under test.

### 4.a Every built-in icon

Ten radio-style buttons on one column (`icon`).

Check:
- [ ] ✓ green, ✗ red, ? yellow, ∅ grey, ⚑ orange, ★ yellow-on-grey, ☑ / ☐ / + / − grey
- [ ] hovering shows the `jba:` name (the `label` tooltip)
- [ ] click one: blue border; click another: the border moves
- [ ] click the selected one again: deselected (column back to empty)
- [ ] Tab to a button and press Enter: same as clicking it (selects it; does **not** submit)
- [ ] Submit, then check `icon` in [Output](#6.-Output)

In [30]:
BUILTINS = ['check', 'x', 'unsure', 'na', 'flag', 'star', 'checked', 'unchecked', 'plus', 'minus']

test = {
    'form_config': {
        'form': [
            {'button': {'column': 'icon', 'value': name, 'icon': f'jba:{name}', 'label': f'jba:{name}'}}
            for name in BUILTINS
        ],
    },
}
jba(**test);

<IPython.core.display.Javascript object>

### 4.b Colours and sizes

Check:
- [ ] 1: built-in ✓ with a dark-blue fill (`background` overrides the green)
- [ ] 2: built-in ✗ with a yellow icon (`color` overrides the dark icon)
- [ ] 3: plain text "A" in pink on teal
- [ ] 1–3: the default 25px square; 4: `size: 18` → smaller; 5: `size: 50` → bigger; 6: `width: 90, height: 25` → wide
- [ ] the text size follows the button size

In [38]:
test = {
    'form_config': {
        'form': [
            {'button': {'column': 'pick', 'value': 1, 'icon': 'jba:check', 'background': '#1e3a8a'}},
            {'button': {'column': 'pick', 'value': 2, 'icon': 'jba:x', 'color': '#f9e2af'}},
            {'button': {'column': 'pick', 'value': 3, 'icon': 'A', 'color': '#f5c2e7', 'background': '#14b8a6'}},
            {'button': {'column': 'pick', 'value': 4, 'icon': 'S', 'size': 18}},
            {'button': {'column': 'pick', 'value': 5, 'icon': 'L', 'size': 50}},
            {'button': {'column': 'pick', 'value': 6, 'icon': 'wide', 'width': 90, 'height': 25}},
        ],
    },
}
jba(**test);

<IPython.core.display.Javascript object>

### 4.c `initial_value` and `initial_icon`

- `present`: two buttons, `initial_value: 'unknown'`: the column starts as "unknown", not empty
- `keep`: one button with `initial_icon: jba:unchecked`: looks like ☐ until clicked, then ☑

Check:
- [ ] neither `present` button is selected at first; Submit → `present` = unknown in [Output](#6.-Output)
- [ ] click **+**: selected; click it again: back to unknown (not empty)
- [ ] `keep` shows ☐; click → ☑ (value 1); click again → ☐ (empty)
- [ ] **Next ▶**: everything back to its starting state

In [43]:
test = {
    'form_config': {
        'form': [
            {'text': 'present:'},
            {'button': {'column': 'present', 'value': 'yes', 'icon': 'jba:plus', 'initial_value': 'unknown'}},
            {'button': {'column': 'present', 'value': 'no', 'icon': 'jba:minus'}},
            {'line': True},
            {'text': 'keep:'},
            {'button': {'column': 'keep', 'value': 1, 'icon': 'jba:checked', 'initial_icon': 'jba:unchecked'}},
        ],
    },
}
jba(**test);

<IPython.core.display.Javascript object>

### 4.d Toggles

| column | setup | starts |
|---|---|---|
| `checkbox` | 2 states, `initial_value: 0` | on ☐ (0) — the state whose value matches |
| `yes_no_unsure` | 3 states, no initial | on the first state (✓ yes) |
| `unset` | 2 states, `initial_value: null` | faded ☑, value empty; first click → ☑ (1) |
| `unset_icon` | 2 states, `initial_icon: jba:na` | ∅, value empty; first click → first state |

Check:
- [ ] each starts as in the table
- [ ] each click moves to the next state and wraps around (✓ → ✗ → ? → ✓)
- [ ] no blue selection border on toggles (the icon is the state)
- [ ] Submit without clicking: `checkbox` = 0, `yes_no_unsure` = yes, the other two empty
- [ ] **Next ▶** resets all four
- [ ] Tab to a toggle and press Enter: it moves to its next state (does **not** submit)

In [ ]:
CHECKBOX_STATES = [{'value': 1, 'icon': 'jba:checked'}, {'value': 0, 'icon': 'jba:unchecked'}]

test = {
    'form_config': {
        'form': [
            {'button': {'column': 'checkbox', 'initial_value': 0, 'toggle': CHECKBOX_STATES}},
            {'button': {'column': 'yes_no_unsure', 'toggle': [
                {'value': 'yes', 'icon': 'jba:check'},
                {'value': 'no', 'icon': 'jba:x'},
                {'value': 'unsure', 'icon': 'jba:unsure'},
            ]}},
            {'button': {'column': 'unset', 'initial_value': None, 'toggle': CHECKBOX_STATES}},
            {'button': {'column': 'unset_icon', 'initial_icon': 'jba:na', 'toggle': CHECKBOX_STATES}},
        ],
    },
}
jba(**test);

### 4.e Submit buttons and `required`

- ✓ / ✗: `submit` + `column` / `value`, `required`: one-click review (writes `verdict`, submits, moves on)
- ⚑: `submit` only (no column)
- `notes` is required: every submit button waits for it
- `verdict` is required: ⚑ and the plain **Submit** wait for it too, but ✓ / ✗ don't (clicking them fills it)

Check:
- [ ] at first ✓, ✗, ⚑ and Submit are all faded / disabled
- [ ] type notes: ✓ and ✗ enable; ⚑ and Submit stay disabled
- [ ] ✓ saves `verdict = yes` (with the notes) and moves to the next clip; ✗ saves `no`
- [ ] Tab to ✓ and press Enter: same as clicking it (saves + next clip)
- [ ] Enter in the notes box does nothing special; Enter on Skip skips (doesn't submit)

In [48]:
test = {
    'form_config': {
        'form': [
            {'textbox': {'label': 'notes', 'required': True}},
            {'line': True},
            {'button': {'column': 'verdict', 'value': 'yes', 'icon': 'jba:check', 'submit': True,
                        'required': True, 'label': 'valid + submit'}},
            {'button': {'column': 'verdict', 'value': 'no', 'icon': 'jba:x', 'submit': True,
                        'label': 'invalid + submit'}},
            {'button': {'icon': 'jba:flag', 'submit': True, 'label': 'just submit'}},
        ],
    },
}
jba(**test);

<IPython.core.display.Javascript object>

### 4.f Image buttons

Images are files read by the kernel ([`annotator_config/icons/`](annotator_config/icons)).

Check:
- [ ] `species`: bird and frog images; select one → blue border
- [ ] `species` buttons use `initial_image` (dashed circle) until selected
- [ ] `has_bird`: a toggle between the bird image and ☐
- [ ] a missing image (`nope.svg`) shows its label ("missing") and a ❌ in the status line

In [62]:
ICONS = 'annotator_config/icons'

test = {
    'form_config': {
        'form': [
            {'button': {'column': 'species', 'value': 'bird', 'image': f'{ICONS}/bird.svg',
                        'initial_image': f'{ICONS}/blank.svg', 'label': 'bird', 'size': 60}},
            {'button': {'column': 'species', 'value': 'frog', 'image': f'{ICONS}/frog.svg',
                        'initial_image': f'{ICONS}/blank.svg', 'label': 'frog', 'size': 60}},
            {'line': True},
            {'button': {'column': 'has_bird', 'initial_value': 0, 'toggle': [
                {'value': 1, 'image': f'{ICONS}/bird.svg'},
                {'value': 0, 'icon': 'jba:unchecked'},
            ]}},
            {'button': {'column': 'broken', 'value': 1, 'image': f'{ICONS}/nope.svg', 'label': 'missing'}},
        ],
    },
}
jba(**test);

<IPython.core.display.Javascript object>

### 4.f+ Enter on a checkbox

Check:
- [ ] Tab to the checkbox and press Enter: it toggles (same as a click) and does **not** submit
- [ ] Space still toggles it too
- [ ] Tab to **Submit** and press Enter: submits

In [ ]:
test = {
    'form_config': {
        'form': [
            {'checkbox': {'label': 'flag for expert review', 'column': 'flagged',
                          'checked_value': 'yes', 'unchecked_value': 'no'}},
        ],
    },
}
jba(**test);

### 4.g Buttons in a dynamic form

Check:
- [ ] "is valid?" = no shows the correction form with its buttons
- [ ] the buttons work there; switching back to yes hides them (their values are still reset per clip)

In [70]:
test = {
    'form_config': {
        'form': [
            {'select': {'label': 'is valid?', 'column': 'is_valid', 'required': True,
                        'items': [{'value': 'yes'}, {'value': 'no', 'form': 'correction'}]}},
        ],
        'dynamic_forms': {
            'correction': [
                {'text': 'what is it?'},
                {'button': {'column': 'correction', 'value': 'noise', 'icon': 'jba:na', 'label': 'noise'}},
                {'button': {'column': 'correction', 'value': 'other species', 'icon': 'jba:unsure', 'label': 'other species'}},
            ],
        },
    },
}
jba(**test);

<IPython.core.display.Javascript object>

---
## 5. Validation

A form with one mistake per button. Each should be reported (no widget is opened).

In [75]:
bad_form = {
    'form': [
        {'button': {'column': 'a'}},                                       # no icon/image; column without value
        {'button': {'icon': 'x'}},                                         # does nothing
        {'button': {'icon': 'jba:nope', 'submit': True}},                  # unknown built-in
        {'button': {'icon': 'x', 'submit': True, 'color': 3}},             # color not a string
        {'button': {'column': 't', 'submit': True,                         # toggle + submit, 1 state
                    'toggle': [{'value': 1, 'icon': 'a'}]}},
        {'button': {'column': 't2', 'toggle': [{'icon': 'a'}, {'value': 0}]}},  # state missing value / icon
    ],
}
result = BioacousticAnnotator(config=CONFIG, form_config=bad_form).validate()
print('valid:', result['valid'])
for msg in result['errors']:
    print('❌', msg)

valid: False
❌ Button "a": set "icon" (text, a symbol or a jba: icon) or "image" (a path)
❌ Button "a": "column" needs a "value" to write
❌ Button "x": set "column" + "value", "submit: true", or both
❌ Button "jba:nope" icon: unknown built-in icon "jba:nope" (use jba:check, jba:checked, jba:flag, jba:minus, jba:na, jba:plus, jba:star, jba:unchecked, jba:unsure, jba:x)
❌ Button "x": "color" must be a CSS color string
❌ Button "t": a toggle can't "submit" (add a separate submit button)
❌ Button "t": "toggle" must be a list of 2 or more states
❌ Button "t2" toggle state 1: needs a "value"
❌ Button "t2" toggle state 2: set "icon" or "image"


---
## 6. Output

The rows submitted from every widget above (except 1.d, which writes its own file). Button
columns hold the button values; toggles their state values; untouched buttons with no
`initial_value` are empty.

In [76]:
ba_defaults.output(force=True).tail(20)

,detection_id,is_valid,call_type,flagged,icon,notes,verdict
0,5780,unsure,call,True,NaN,NaN,NaN
1,5780,NaN,NaN,NaN,plus,NaN,NaN
2,5780,NaN,NaN,NaN,NaN,asd,yes
3,1191,NaN,NaN,NaN,NaN,SDAF,no
4,5429,NaN,NaN,NaN,NaN,AFDSA,no
